# Scaling Up: From Ollama to vLLM with Granite 3.3

This notebook provides a comprehensive guide for scaling from Ollama (great for local development and prototyping) to vLLM (optimized for production-grade, high-throughput model serving) using IBM's Granite 3.3 8B model.

## 🎯 What You'll Learn

- **The Scaling Challenge**: Understanding when and why to transition from Ollama to vLLM
- **Performance Comparison**: Key differences between Ollama and vLLM architectures
- **Migration Guide**: Step-by-step process to set up vLLM with Granite 3.3 8B
- **Benchmarking**: Practical performance testing and optimization techniques
- **Production Considerations**: Best practices for deploying vLLM in production environments

## 📋 Prerequisites

- Python 3.8+
- NVIDIA GPU with CUDA support (recommended)
- 16GB+ RAM
- Basic understanding of REST APIs and Python

## 1. The Scaling Challenge: When Local Development Meets Production Demands

### The Journey from Prototype to Production

When you start working with Large Language Models (LLMs), **Ollama** is often the go-to choice for good reasons:
- ✅ **Easy Setup**: Get models running locally in minutes
- ✅ **Developer-Friendly**: Simple CLI with Docker-like commands
- ✅ **Cross-Platform**: Works on macOS, Linux, and Windows
- ✅ **OpenAI-Compatible API**: Easy integration with existing tools

However, as your project grows and requirements evolve, you might encounter these challenges:

### 🚨 Signs It's Time to Scale Up

| **Challenge** | **Ollama Limitation** | **vLLM Solution** |
|---------------|----------------------|-------------------|
| **High Concurrent Users** | Sequential request processing | Continuous batching for parallel requests |
| **Production Workloads** | Single-user optimization | Multi-user, high-throughput design |
| **Long Context Windows** | Memory inefficiency with long prompts | Optimized memory management |
| **Custom Model Integration** | Limited to curated models | Direct HuggingFace integration |
| **Fine-Grained Control** | Limited configuration options | Extensive customization parameters |
| **Enterprise Scaling** | Resource usage not optimized | GPU utilization optimization |

### Why This Migration Matters

Moving from Ollama to vLLM isn't just about performance—it's about **architectural readiness** for production environments where:
- Multiple users need simultaneous access
- Response time consistency is critical
- Resource efficiency directly impacts costs
- Scalability and reliability are non-negotiable

## 2. Architecture Deep Dive: Ollama vs vLLM

### 🏗️ Ollama: The Developer's Friend

**Ollama** is designed for simplicity and ease of use:

```
┌─────────────────────────────────────────┐
│              OLLAMA ARCHITECTURE        │
├─────────────────────────────────────────┤
│  REST API (OpenAI Compatible)          │
├─────────────────────────────────────────┤
│  Model Management Layer                 │
├─────────────────────────────────────────┤
│  llama.cpp Backend                      │
├─────────────────────────────────────────┤
│  Sequential Request Processing          │
└─────────────────────────────────────────┘
```

**Key Characteristics:**
- **Sequential Processing**: One request at a time
- **Model Abstraction**: Curated, pre-configured models
- **Resource Management**: Automatic but not optimized for multi-user scenarios
- **Use Case**: Development, prototyping, single-user applications

### ⚡ vLLM: The Performance Engine

**vLLM** is architected for high-throughput production environments:

```
┌─────────────────────────────────────────┐
│               vLLM ARCHITECTURE         │
├─────────────────────────────────────────┤
│  OpenAI-Compatible API Server          │
├─────────────────────────────────────────┤
│  Request Router & Load Balancer        │
├─────────────────────────────────────────┤
│  Continuous Batching Engine            │
├─────────────────────────────────────────┤
│  PagedAttention Memory Management      │
├─────────────────────────────────────────┤
│  Optimized CUDA Kernels                │
└─────────────────────────────────────────┘
```

**Key Innovations:**
- **Continuous Batching**: Process multiple requests simultaneously
- **PagedAttention**: Efficient memory management for attention mechanisms
- **Dynamic Batching**: Automatic optimization of batch sizes
- **GPU Optimization**: Custom CUDA kernels for maximum throughput

## 3. When and Why to Transition to vLLM

### 📊 Performance Indicators: Is It Time to Scale?

You should consider migrating to vLLM when you notice these patterns:

#### **Throughput Bottlenecks**
- **Observation**: Response times increase significantly under load
- **Ollama Behavior**: Queue builds up as requests are processed sequentially
- **vLLM Solution**: Continuous batching maintains consistent response times

#### **Resource Inefficiency**
- **Observation**: GPU utilization is low despite high demand
- **Ollama Behavior**: GPU idles between requests
- **vLLM Solution**: Keeps GPU saturated with optimized batching

#### **Memory Pressure**
- **Observation**: Out-of-memory errors with longer contexts
- **Ollama Behavior**: Standard attention mechanisms use quadratic memory
- **vLLM Solution**: PagedAttention reduces memory usage by 2-4x

### 🎯 Decision Matrix: Ollama vs vLLM

| **Scenario** | **Choose Ollama** | **Choose vLLM** |
|--------------|-------------------|-----------------|
| **Development & Prototyping** | ✅ Perfect fit | ❌ Overkill |
| **Single User Applications** | ✅ Simple setup | ⚠️ Additional complexity |
| **Multi-User Production** | ❌ Poor performance | ✅ Designed for this |
| **High-Throughput APIs** | ❌ Sequential bottleneck | ✅ Continuous batching |
| **Custom Model Integration** | ⚠️ Limited options | ✅ Direct HF integration |
| **Enterprise Deployment** | ❌ Not production-ready | ✅ Production-optimized |

### 🚀 Migration Triggers

Consider transitioning when you experience:

1. **>10 concurrent users** regularly accessing your model
2. **Response time variability** affecting user experience  
3. **GPU utilization <50%** despite high request volume
4. **Memory limitations** with context windows >4K tokens
5. **Need for custom models** not available in Ollama's catalog
6. **Production SLA requirements** for uptime and performance

## 4. vLLM Setup Guide with Granite 3.3 8B

### 🛠️ Prerequisites Check

Before we begin, let's verify our environment meets the requirements:

In [ ]:
import sys
import subprocess
import platform
import psutil
import torch

def check_prerequisites():
    """Check system prerequisites for vLLM setup"""
    print("🔍 System Prerequisites Check")
    print("=" * 50)
    
    # Python version
    python_version = sys.version_info
    print(f"Python Version: {python_version.major}.{python_version.minor}.{python_version.micro}")
    
    if python_version >= (3, 8):
        print("✅ Python version is compatible")
    else:
        print("❌ Python 3.8+ required")
    
    # System info
    print(f"Operating System: {platform.system()} {platform.release()}")
    
    # Memory check
    memory_gb = psutil.virtual_memory().total / (1024**3)
    print(f"System RAM: {memory_gb:.1f} GB")
    
    if memory_gb >= 16:
        print("✅ Sufficient RAM for Granite 3.3 8B")
    else:
        print("⚠️  Recommended: 16GB+ RAM for optimal performance")
    
    # GPU check
    if torch.cuda.is_available():
        gpu_count = torch.cuda.device_count()
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        
        print(f"GPU: {gpu_name}")
        print(f"GPU Memory: {gpu_memory:.1f} GB")
        print(f"CUDA Version: {torch.version.cuda}")
        print("✅ CUDA GPU detected")
        
        if gpu_memory >= 8:
            print("✅ Sufficient GPU memory for Granite 3.3 8B")
        else:
            print("⚠️  Consider using quantized models for <8GB GPU memory")
    else:
        print("⚠️  No CUDA GPU detected - CPU inference will be slower")
    
    print("\n" + "=" * 50)

# Run the check
check_prerequisites()

### 📦 Step 1: Install vLLM

vLLM can be installed via pip with CUDA support. Choose the appropriate installation method based on your setup:

In [ ]:
# Install vLLM with CUDA support
# Note: This might take a few minutes to complete

import subprocess
import sys

def install_vllm():
    """Install vLLM and required dependencies"""
    print("🚀 Installing vLLM...")
    
    # Core vLLM installation
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", 
            "vllm==0.6.3.post1",  # Latest stable version as of 2024
            "--upgrade"
        ])
        print("✅ vLLM installed successfully")
    except subprocess.CalledProcessError as e:
        print(f"❌ Error installing vLLM: {e}")
        return False
    
    # Additional dependencies for Granite models
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install",
            "transformers>=4.36.0",
            "torch>=2.0.0",
            "huggingface-hub",
            "requests",
            "openai",  # For API client testing
            "--upgrade"
        ])
        print("✅ Additional dependencies installed")
    except subprocess.CalledProcessError as e:
        print(f"❌ Error installing dependencies: {e}")
        return False
    
    return True

print("✅ vLLM installation function ready")
print("💡 To install vLLM, run: install_vllm()")
print("   Or install manually: pip install vllm transformers torch huggingface-hub requests openai")

### 🧠 Step 2: Load Granite 3.3 8B Model

Now let's set up the Granite 3.3 8B model with vLLM. We'll demonstrate both programmatic loading and server setup:

In [ ]:
from vllm import LLM, SamplingParams
import time

# Model configuration
MODEL_NAME = "ibm-granite/granite-3.3-8b-instruct"

def load_granite_model():
    """Load Granite 3.3 8B model with optimized vLLM settings"""
    print("🔄 Loading Granite 3.3 8B model...")
    print("   This may take a few minutes on first run (model download)")
    
    start_time = time.time()
    
    try:
        # Initialize vLLM with optimized settings for Granite
        llm = LLM(
            model=MODEL_NAME,
            tensor_parallel_size=1,  # Adjust based on your GPU count
            gpu_memory_utilization=0.9,  # Use 90% of GPU memory
            max_model_len=8192,  # Context window for Granite 3.3 8B
            enable_prefix_caching=True,  # Optimize for repeated prefixes
            disable_log_stats=False,  # Keep performance logging
            # trust_remote_code=True,  # Uncomment if needed for custom model code
        )
        
        load_time = time.time() - start_time
        print(f"✅ Model loaded successfully in {load_time:.1f} seconds")
        return llm
        
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        print("💡 Tips for troubleshooting:")
        print("   - Ensure you have sufficient GPU memory")
        print("   - Try reducing gpu_memory_utilization to 0.7")
        print("   - Check CUDA installation")
        return None

print("✅ Model loading function ready")
print("💡 To load the Granite model, run: llm = load_granite_model()")
print("   Warning: This will download ~16GB model on first run")

In [ ]:
def test_granite_inference(llm):
    """Test basic inference with Granite 3.3 8B"""
    if llm is None:
        print("❌ Model not loaded. Load the model first.")
        return
    
    print("🧪 Testing Granite 3.3 8B Inference")
    print("=" * 50)
    
    # Sample prompts for testing
    prompts = [
        "What are the key differences between Python and JavaScript?",
        "Explain quantum computing in simple terms.",
        "Write a Python function to calculate the Fibonacci sequence."
    ]
    
    # Configure sampling parameters
    sampling_params = SamplingParams(
        temperature=0.7,
        top_p=0.9,
        max_tokens=200,
        stop=["</s>", "<|endoftext|>"]  # Common stop tokens
    )
    
    start_time = time.time()
    
    # Generate responses
    outputs = llm.generate(prompts, sampling_params)
    
    total_time = time.time() - start_time
    
    # Display results
    for i, output in enumerate(outputs):
        prompt = output.prompt
        generated_text = output.outputs[0].text
        
        print(f"\n📝 Prompt {i+1}: {prompt[:50]}...")
        print(f"🤖 Response: {generated_text}")
        print("-" * 30)
    
    print(f"\n⏱️  Total generation time: {total_time:.2f} seconds")
    print(f"📊 Average time per prompt: {total_time/len(prompts):.2f} seconds")

print("✅ Inference testing function ready")
print("💡 To test inference, run: test_granite_inference(llm)")
print("   Make sure to load the model first")

### 🚀 Step 3: Setting Up vLLM API Server

For production use, you'll want to run vLLM as an API server. This provides an OpenAI-compatible endpoint that can handle multiple concurrent requests:

In [ ]:
import subprocess
import time
import requests
import json

class vLLMServer:
    """Wrapper class to manage vLLM API server"""
    
    def __init__(self, model_name=MODEL_NAME, port=8000):
        self.model_name = model_name
        self.port = port
        self.process = None
        self.base_url = f"http://localhost:{port}"
    
    def start_server(self):
        """Start vLLM API server in background"""
        print(f"🚀 Starting vLLM server on port {self.port}...")
        
        # Server command
        cmd = [
            "python", "-m", "vllm.entrypoints.openai.api_server",
            "--model", self.model_name,
            "--port", str(self.port),
            "--gpu-memory-utilization", "0.9",
            "--max-model-len", "8192",
            "--enable-prefix-caching",
        ]
        
        try:
            # Start server process
            self.process = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True
            )
            
            # Wait for server to start
            print("⏳ Waiting for server to initialize...")
            self._wait_for_server()
            print("✅ vLLM server is running!")
            return True
            
        except Exception as e:
            print(f"❌ Error starting server: {e}")
            return False
    
    def _wait_for_server(self, timeout=120):
        """Wait for server to be ready"""
        start_time = time.time()
        while time.time() - start_time < timeout:
            try:
                response = requests.get(f"{self.base_url}/v1/models", timeout=5)
                if response.status_code == 200:
                    return True
            except:
                pass
            time.sleep(2)
        raise TimeoutError("Server failed to start within timeout")
    
    def test_api(self):
        """Test the API with a simple request"""
        if not self.is_running():
            print("❌ Server is not running")
            return
        
        print("🧪 Testing API endpoint...")
        
        headers = {"Content-Type": "application/json"}
        payload = {
            "model": self.model_name,
            "messages": [
                {"role": "user", "content": "What is the capital of France?"}
            ],
            "max_tokens": 100,
            "temperature": 0.7
        }
        
        try:
            response = requests.post(
                f"{self.base_url}/v1/chat/completions",
                headers=headers,
                json=payload,
                timeout=30
            )
            
            if response.status_code == 200:
                result = response.json()
                message = result['choices'][0]['message']['content']
                print(f"✅ API Response: {message}")
            else:
                print(f"❌ API Error: {response.status_code} - {response.text}")
                
        except Exception as e:
            print(f"❌ Request failed: {e}")
    
    def is_running(self):
        """Check if server is running"""
        try:
            response = requests.get(f"{self.base_url}/health", timeout=5)
            return response.status_code == 200
        except:
            return False
    
    def stop_server(self):
        """Stop the server"""
        if self.process:
            self.process.terminate()
            self.process.wait()
            print("🛑 Server stopped")

# Create server instance
server = vLLMServer()

print("💡 Use server.start_server() to start the API server")
print("   Then use server.test_api() to test it")
print("   Use server.stop_server() to shut it down")
print("\n⚠️  Note: Starting the server will take several minutes and use significant GPU memory")

In [ ]:
# Ollama Integration for Comparison Testing
import requests
import time

class OllamaClient:
    """Simple client for Ollama API testing"""
    
    def __init__(self, base_url="http://localhost:11434", model_name="granite3.3:8b"):
        self.base_url = base_url
        self.model_name = model_name
    
    def is_running(self):
        """Check if Ollama server is running"""
        try:
            response = requests.get(f"{self.base_url}/api/tags", timeout=5)
            return response.status_code == 200
        except:
            return False
    
    def test_inference(self, prompt, max_tokens=200):
        """Test a single inference request"""
        if not self.is_running():
            print("❌ Ollama server is not running")
            return None
        
        payload = {
            "model": self.model_name,
            "prompt": prompt,
            "stream": False,
            "options": {
                "num_predict": max_tokens,
                "temperature": 0.7
            }
        }
        
        start_time = time.time()
        try:
            response = requests.post(
                f"{self.base_url}/api/generate",
                json=payload,
                timeout=60
            )
            
            if response.status_code == 200:
                result = response.json()
                total_time = time.time() - start_time
                
                return {
                    'success': True,
                    'response': result['response'],
                    'total_time': total_time,
                    'eval_count': result.get('eval_count', 0),
                    'eval_duration': result.get('eval_duration', 0) / 1e9 if result.get('eval_duration') else 0,
                    'tokens_per_second': result.get('eval_count', 0) / (result.get('eval_duration', 1) / 1e9) if result.get('eval_duration') else 0
                }
            else:
                return {'success': False, 'error': f"HTTP {response.status_code}"}
        except Exception as e:
            return {'success': False, 'error': str(e)}

# Create Ollama client instance
ollama = OllamaClient()

print("🦙 Ollama client ready!")
if ollama.is_running():
    print("✅ Ollama server is running")
else:
    print("⚠️  Ollama server not detected. Start it with: ollama serve")
    print("   Make sure granite3.3:8b model is available: ollama pull granite3.3:8b")

In [ ]:
# Comprehensive Testing: Ollama vs vLLM
import time

def test_ollama_inference():
    """Test Ollama with granite3.3:8b model"""
    print("🦙 Testing Ollama Performance")
    print("=" * 50)
    
    test_prompts = [
        "What is machine learning?",
        "Explain Python in one sentence.",
        "What are the benefits of cloud computing?"
    ]
    
    if not ollama.is_running():
        print("❌ Ollama server not running")
        return None
    
    results = []
    total_start = time.time()
    
    for i, prompt in enumerate(test_prompts):
        print(f"\n📝 Testing prompt {i+1}: {prompt[:30]}...")
        result = ollama.test_inference(prompt, max_tokens=100)
        
        if result and result['success']:
            print(f"✅ Response time: {result['total_time']:.2f}s")
            print(f"🚀 Tokens/sec: {result['tokens_per_second']:.1f}")
            print(f"📄 Response: {result['response'][:100]}...")
            results.append(result)
        else:
            print(f"❌ Failed: {result.get('error', 'Unknown error') if result else 'No response'}")
    
    total_time = time.time() - total_start
    
    if results:
        avg_time = sum(r['total_time'] for r in results) / len(results)
        avg_tokens_per_sec = sum(r['tokens_per_second'] for r in results) / len(results)
        
        print(f"\n📊 Ollama Summary:")
        print(f"   Average response time: {avg_time:.2f}s")
        print(f"   Average tokens/sec: {avg_tokens_per_sec:.1f}")
        print(f"   Total test time: {total_time:.2f}s")
        print(f"   Success rate: {len(results)}/{len(test_prompts)} ({len(results)/len(test_prompts)*100:.1f}%)")
        
        return {
            'platform': 'Ollama',
            'avg_response_time': avg_time,
            'avg_tokens_per_sec': avg_tokens_per_sec,
            'total_time': total_time,
            'success_rate': len(results)/len(test_prompts)*100,
            'successful_tests': len(results)
        }
    else:
        print("❌ All tests failed")
        return None

def test_vllm_offline_inference():
    """Test vLLM programmatic inference (without server)"""
    print("\n⚡ Testing vLLM Offline Performance")
    print("=" * 50)
    
    try:
        from vllm import LLM, SamplingParams
        
        # Try to load model with reduced memory requirements
        print("🔄 Loading Granite model with vLLM...")
        llm = LLM(
            model=MODEL_NAME,
            tensor_parallel_size=1,
            gpu_memory_utilization=0.8,  # Reduced for CPU
            max_model_len=2048,  # Reduced context for CPU
            enforce_eager=True,  # Better for CPU
            disable_log_stats=True
        )
        
        test_prompts = [
            "What is machine learning?",
            "Explain Python in one sentence.",
            "What are the benefits of cloud computing?"
        ]
        
        sampling_params = SamplingParams(
            temperature=0.7,
            top_p=0.9,
            max_tokens=100
        )
        
        print("🧪 Running vLLM inference tests...")
        start_time = time.time()
        outputs = llm.generate(test_prompts, sampling_params)
        total_time = time.time() - start_time
        
        print(f"\n📊 vLLM Results:")
        for i, output in enumerate(outputs):
            response = output.outputs[0].text
            print(f"✅ Prompt {i+1}: {response[:100]}...")
        
        avg_time_per_prompt = total_time / len(test_prompts)
        print(f"\n📊 vLLM Summary:")
        print(f"   Total generation time: {total_time:.2f}s")
        print(f"   Average time per prompt: {avg_time_per_prompt:.2f}s")
        print(f"   Success rate: 100%")
        
        return {
            'platform': 'vLLM',
            'avg_response_time': avg_time_per_prompt,
            'total_time': total_time,
            'success_rate': 100.0,
            'successful_tests': len(test_prompts)
        }
        
    except Exception as e:
        print(f"❌ vLLM test failed: {e}")
        print("💡 This might be due to CPU-only mode or insufficient memory")
        return None

# Run comprehensive comparison
print("🚀 Starting Ollama vs vLLM Comparison Test")
print("=" * 60)

# Test Ollama
ollama_results = test_ollama_inference()

# Test vLLM (offline mode)
vllm_results = test_vllm_offline_inference()

# Compare results
print("\n📈 Performance Comparison")
print("=" * 60)

if ollama_results and vllm_results:
    print(f"🦙 Ollama - Avg Response Time: {ollama_results['avg_response_time']:.2f}s")
    print(f"⚡ vLLM   - Avg Response Time: {vllm_results['avg_response_time']:.2f}s")
    
    speedup = ollama_results['avg_response_time'] / vllm_results['avg_response_time']
    print(f"\n🚀 vLLM is {speedup:.1f}x {'faster' if speedup > 1 else 'slower'} than Ollama")
    
elif ollama_results:
    print("✅ Ollama test completed successfully")
    print("❌ vLLM test failed - may require GPU or more memory")
    
elif vllm_results:
    print("❌ Ollama test failed - server may not be running")
    print("✅ vLLM test completed successfully")
    
else:
    print("❌ Both tests failed - check your setup")

print("\n💡 Notes:")
print("   • Ollama optimized for ease of use and quick setup")
print("   • vLLM optimized for throughput and production workloads")
print("   • Performance varies based on hardware and model size")

## 5. Performance Benchmarking and Comparisons

### 📊 Understanding Key Performance Metrics

Before we benchmark, let's understand the key metrics that matter for LLM serving:

- **Throughput**: Requests processed per second (RPS)
- **Latency**: Time to first token (TTFT) and time per output token (TPOT)
- **Concurrent Users**: Maximum simultaneous users supported
- **Resource Utilization**: GPU memory and compute efficiency
- **Quality**: Consistency of outputs under load

In [ ]:
import asyncio
import aiohttp
import time
import statistics
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

class LLMBenchmark:
    """Comprehensive benchmarking suite for LLM servers"""
    
    def __init__(self, base_url="http://localhost:8000", model_name=MODEL_NAME):
        self.base_url = base_url
        self.model_name = model_name
        self.results = {}
    
    async def single_request(self, session, prompt, max_tokens=100):
        """Make a single API request and measure timing"""
        payload = {
            "model": self.model_name,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
            "temperature": 0.7
        }
        
        start_time = time.time()
        try:
            async with session.post(
                f"{self.base_url}/v1/chat/completions",
                json=payload,
                timeout=aiohttp.ClientTimeout(total=60)
            ) as response:
                if response.status == 200:
                    result = await response.json()
                    end_time = time.time()
                    
                    # Extract metrics
                    total_time = end_time - start_time
                    output_tokens = len(result['choices'][0]['message']['content'].split())
                    
                    return {
                        'success': True,
                        'total_time': total_time,
                        'output_tokens': output_tokens,
                        'tokens_per_second': output_tokens / total_time if total_time > 0 else 0,
                        'response': result['choices'][0]['message']['content']
                    }
                else:
                    return {'success': False, 'error': f"HTTP {response.status}"}
        except Exception as e:
            return {'success': False, 'error': str(e)}
    
    async def concurrent_benchmark(self, prompts, concurrent_users=5):
        """Benchmark concurrent request handling"""
        print(f"🔥 Running concurrent benchmark with {concurrent_users} users...")
        
        async with aiohttp.ClientSession() as session:
            tasks = []
            
            # Create tasks for concurrent requests
            for i in range(concurrent_users):
                prompt = prompts[i % len(prompts)]
                task = self.single_request(session, prompt)
                tasks.append(task)
            
            # Execute all requests concurrently
            start_time = time.time()
            results = await asyncio.gather(*tasks, return_exceptions=True)
            total_time = time.time() - start_time
            
            # Analyze results
            successful_requests = [r for r in results if isinstance(r, dict) and r.get('success')]
            failed_requests = len(results) - len(successful_requests)
            
            if successful_requests:
                avg_response_time = statistics.mean([r['total_time'] for r in successful_requests])
                avg_tokens_per_sec = statistics.mean([r['tokens_per_second'] for r in successful_requests])
                total_throughput = len(successful_requests) / total_time
                
                return {
                    'concurrent_users': concurrent_users,
                    'total_requests': len(results),
                    'successful_requests': len(successful_requests),
                    'failed_requests': failed_requests,
                    'total_time': total_time,
                    'avg_response_time': avg_response_time,
                    'avg_tokens_per_sec': avg_tokens_per_sec,
                    'requests_per_second': total_throughput,
                    'success_rate': len(successful_requests) / len(results) * 100
                }
            else:
                return {'error': 'All requests failed'}
    
    def sequential_benchmark(self, prompts, iterations=10):
        """Benchmark sequential request processing"""
        print(f"📊 Running sequential benchmark with {iterations} iterations...")
        
        results = []
        
        for i in range(iterations):
            prompt = prompts[i % len(prompts)]
            
            payload = {
                "model": self.model_name,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 100,
                "temperature": 0.7
            }
            
            start_time = time.time()
            try:
                response = requests.post(
                    f"{self.base_url}/v1/chat/completions",
                    json=payload,
                    timeout=60
                )
                
                if response.status_code == 200:
                    result = response.json()
                    end_time = time.time()
                    
                    total_time = end_time - start_time
                    output_tokens = len(result['choices'][0]['message']['content'].split())
                    
                    results.append({
                        'iteration': i + 1,
                        'total_time': total_time,
                        'output_tokens': output_tokens,
                        'tokens_per_second': output_tokens / total_time if total_time > 0 else 0
                    })
                else:
                    print(f"❌ Request {i+1} failed: {response.status_code}")
            
            except Exception as e:
                print(f"❌ Request {i+1} error: {e}")
        
        if results:
            avg_time = statistics.mean([r['total_time'] for r in results])
            avg_tokens_per_sec = statistics.mean([r['tokens_per_second'] for r in results])
            
            return {
                'iterations': iterations,
                'successful_requests': len(results),
                'avg_response_time': avg_time,
                'avg_tokens_per_sec': avg_tokens_per_sec,
                'throughput': len(results) / sum([r['total_time'] for r in results]),
                'detailed_results': results
            }
        else:
            return {'error': 'All requests failed'}
    
    def plot_results(self, concurrent_results, sequential_results):
        """Create visualization of benchmark results"""
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
        
        # Plot 1: Response Time Comparison
        if concurrent_results and 'avg_response_time' in concurrent_results:
            categories = ['Sequential', 'Concurrent']
            response_times = [
                sequential_results.get('avg_response_time', 0),
                concurrent_results.get('avg_response_time', 0)
            ]
            ax1.bar(categories, response_times, color=['blue', 'orange'])
            ax1.set_title('Average Response Time')
            ax1.set_ylabel('Seconds')
        
        # Plot 2: Throughput Comparison
        if concurrent_results and 'requests_per_second' in concurrent_results:
            throughput_data = [
                sequential_results.get('throughput', 0),
                concurrent_results.get('requests_per_second', 0)
            ]
            ax2.bar(categories, throughput_data, color=['green', 'red'])
            ax2.set_title('Throughput (Requests/Second)')
            ax2.set_ylabel('RPS')
        
        # Plot 3: Tokens per Second
        if concurrent_results and 'avg_tokens_per_sec' in concurrent_results:
            tokens_data = [
                sequential_results.get('avg_tokens_per_sec', 0),
                concurrent_results.get('avg_tokens_per_sec', 0)
            ]
            ax3.bar(categories, tokens_data, color=['purple', 'cyan'])
            ax3.set_title('Tokens per Second')
            ax3.set_ylabel('Tokens/Second')
        
        # Plot 4: Success Rate
        if concurrent_results and 'success_rate' in concurrent_results:
            success_rates = [100, concurrent_results.get('success_rate', 0)]
            ax4.bar(categories, success_rates, color=['lightgreen', 'lightcoral'])
            ax4.set_title('Success Rate')
            ax4.set_ylabel('Percentage')
            ax4.set_ylim(0, 100)
        
        plt.tight_layout()
        plt.show()

# Test prompts for benchmarking
test_prompts = [
    "Explain the concept of machine learning in 100 words.",
    "What are the benefits of using cloud computing?",
    "How does blockchain technology work?",
    "Describe the process of photosynthesis.",
    "What is the difference between AI and machine learning?",
    "Explain quantum computing to a beginner.",
    "What are the advantages of renewable energy?",
    "How do neural networks learn?",
    "What is the future of electric vehicles?",
    "Explain the importance of cybersecurity."
]

# Create benchmark instance
benchmark = LLMBenchmark()

print("🎯 Benchmark toolkit ready!")
print("📋 Available methods:")
print("   - benchmark.sequential_benchmark(test_prompts)")
print("   - await benchmark.concurrent_benchmark(test_prompts, concurrent_users=5)")
print("   - benchmark.plot_results(concurrent_results, sequential_results)")
print("\n⚠️  Make sure vLLM server is running before benchmarking")

In [ ]:
# Example benchmark execution
async def run_comprehensive_benchmark():
    """Run a comprehensive benchmark comparing different scenarios"""
    
    if not benchmark.base_url:
        print("❌ vLLM server not configured. Set up server first.")
        return
    
    print("🚀 Starting Comprehensive Benchmark Suite")
    print("=" * 60)
    
    # Test 1: Sequential Performance
    print("\n📊 Test 1: Sequential Performance")
    sequential_results = benchmark.sequential_benchmark(test_prompts, iterations=5)
    
    if 'error' not in sequential_results:
        print(f"✅ Sequential Test Results:")
        print(f"   Average Response Time: {sequential_results['avg_response_time']:.2f}s")
        print(f"   Average Tokens/Second: {sequential_results['avg_tokens_per_sec']:.1f}")
        print(f"   Throughput: {sequential_results['throughput']:.2f} RPS")
    
    # Test 2: Concurrent Performance (Light Load)
    print("\n🔥 Test 2: Concurrent Performance (5 users)")
    concurrent_5_results = await benchmark.concurrent_benchmark(test_prompts, concurrent_users=5)
    
    if 'error' not in concurrent_5_results:
        print(f"✅ Concurrent Test Results (5 users):")
        print(f"   Success Rate: {concurrent_5_results['success_rate']:.1f}%")
        print(f"   Average Response Time: {concurrent_5_results['avg_response_time']:.2f}s")
        print(f"   Requests per Second: {concurrent_5_results['requests_per_second']:.2f}")
        print(f"   Average Tokens/Second: {concurrent_5_results['avg_tokens_per_sec']:.1f}")
    
    # Test 3: Concurrent Performance (Heavy Load)
    print("\n🔥 Test 3: Concurrent Performance (10 users)")
    concurrent_10_results = await benchmark.concurrent_benchmark(test_prompts, concurrent_users=10)
    
    if 'error' not in concurrent_10_results:
        print(f"✅ Concurrent Test Results (10 users):")
        print(f"   Success Rate: {concurrent_10_results['success_rate']:.1f}%")
        print(f"   Average Response Time: {concurrent_10_results['avg_response_time']:.2f}s")
        print(f"   Requests per Second: {concurrent_10_results['requests_per_second']:.2f}")
        print(f"   Average Tokens/Second: {concurrent_10_results['avg_tokens_per_sec']:.1f}")
    
    # Generate comparison visualization
    print("\n📈 Generating Performance Visualization...")
    benchmark.plot_results(concurrent_5_results, sequential_results)
    
    # Summary and recommendations
    print("\n📋 Performance Summary & Recommendations")
    print("=" * 60)
    
    if 'error' not in sequential_results and 'error' not in concurrent_5_results:
        speedup = concurrent_5_results['requests_per_second'] / sequential_results['throughput']
        print(f"🚀 Concurrent vs Sequential Speedup: {speedup:.1f}x")
        
        if speedup > 3:
            print("✅ Excellent parallelization - vLLM is highly effective")
        elif speedup > 2:
            print("✅ Good parallelization - significant improvement over sequential")
        else:
            print("⚠️  Limited parallelization - check GPU memory and model config")
    
    return {
        'sequential': sequential_results,
        'concurrent_5': concurrent_5_results,
        'concurrent_10': concurrent_10_results
    }

# Comparison with typical Ollama performance
def show_ollama_comparison():
    """Show typical performance differences between Ollama and vLLM"""
    
    print("📊 Typical Performance Comparison: Ollama vs vLLM")
    print("=" * 60)
    
    comparison_data = {
        'Metric': [
            'Sequential Latency (8B model)',
            'Concurrent Users (no degradation)',
            'Memory Efficiency',
            'GPU Utilization',
            'Throughput (requests/second)',
            'Setup Complexity'
        ],
        'Ollama': [
            '2-4 seconds',
            '1-2 users',
            'Standard',
            '60-70%',
            '0.3-0.5 RPS',
            'Very Low'
        ],
        'vLLM': [
            '1-2 seconds',
            '5-20+ users',
            'Optimized (2-4x better)',
            '85-95%',
            '2-10+ RPS',
            'Medium'
        ],
        'Improvement': [
            '2x faster',
            '10x more users',
            '2-4x memory savings',
            '20-35% better',
            '5-20x throughput',
            'Worth the complexity'
        ]
    }
    
    df = pd.DataFrame(comparison_data)
    print(df.to_string(index=False))
    
    print("\n💡 Key Takeaways:")
    print("   • vLLM excels in multi-user, production scenarios")
    print("   • Ollama is perfect for development and single-user use")
    print("   • Migration is justified when serving >5 concurrent users")
    print("   • Memory efficiency becomes crucial with larger models")

# Show the comparison
show_ollama_comparison()

print("\n🎯 To run the full benchmark:")
print("   results = await run_comprehensive_benchmark()")
print("\n⚠️  Ensure vLLM server is running and responsive before benchmarking")

## 6. Production Considerations and Optimization

### 🏗️ Production Deployment Best Practices

Deploying vLLM in production requires careful consideration of several factors:

#### **Infrastructure Requirements**

| **Component** | **Minimum** | **Recommended** | **Optimal** |
|---------------|-------------|-----------------|-------------|
| **GPU Memory** | 8GB | 16GB | 24GB+ |
| **System RAM** | 16GB | 32GB | 64GB+ |
| **CPU Cores** | 4 | 8 | 16+ |
| **Storage** | 50GB SSD | 100GB NVMe | 500GB+ NVMe |
| **Network** | 1Gbps | 10Gbps | 25Gbps+ |

#### **Configuration Optimization**

| **Parameter** | **Description** | **Granite 3.3 8B Recommendation** |
|---------------|----------------|-----------------------------------|
| `gpu_memory_utilization` | GPU memory usage fraction | 0.85-0.9 (leave headroom) |
| `max_model_len` | Maximum context length | 8192 (Granite's native) |
| `tensor_parallel_size` | Multi-GPU parallelism | 1 (for single GPU) |
| `max_num_seqs` | Max concurrent sequences | 128-256 |
| `enable_prefix_caching` | Cache common prefixes | True |
| `swap_space` | CPU offload space | 4-8GB |

In [ ]:
import psutil
from dataclasses import dataclass
import json

try:
    import GPUtil
    GPUTIL_AVAILABLE = True
except ImportError:
    GPUTIL_AVAILABLE = False

@dataclass
class OptimizationRecommendation:
    """Data class for optimization recommendations"""
    category: str
    issue: str
    recommendation: str
    priority: str  # High, Medium, Low
    expected_improvement: str

class vLLMOptimizer:
    """Production optimization analyzer for vLLM deployments"""
    
    def __init__(self):
        self.recommendations = []
    
    def analyze_system_resources(self):
        """Analyze system resources and provide optimization recommendations"""
        print("🔍 Analyzing System Resources for vLLM Optimization")
        print("=" * 60)
        
        # Memory Analysis
        memory = psutil.virtual_memory()
        memory_gb = memory.total / (1024**3)
        memory_usage = memory.percent
        
        print(f"💾 System Memory: {memory_gb:.1f}GB (Usage: {memory_usage:.1f}%)")
        
        if memory_gb < 32:
            self.recommendations.append(OptimizationRecommendation(
                category="Memory",
                issue="Insufficient system RAM",
                recommendation="Upgrade to 32GB+ RAM for better performance",
                priority="Medium",
                expected_improvement="20-30% better throughput"
            ))
        
        # GPU Analysis
        if GPUTIL_AVAILABLE:
            try:
                gpus = GPUtil.getGPUs()
                if gpus:
                    for i, gpu in enumerate(gpus):
                        print(f"🎮 GPU {i}: {gpu.name}")
                        print(f"   Memory: {gpu.memoryUsed}MB / {gpu.memoryTotal}MB ({gpu.memoryUtil*100:.1f}%)")
                        print(f"   Utilization: {gpu.load*100:.1f}%")
                        
                        if gpu.memoryTotal < 8000:  # Less than 8GB
                            self.recommendations.append(OptimizationRecommendation(
                                category="GPU",
                                issue="Limited GPU memory",
                                recommendation="Consider quantized models or upgrade GPU",
                                priority="High",
                                expected_improvement="Enable larger batch sizes"
                            ))
                        
                        if gpu.load < 0.7 and gpu.memoryUtil > 0.8:
                            self.recommendations.append(OptimizationRecommendation(
                                category="GPU",
                                issue="Memory bound, not compute bound",
                                recommendation="Reduce max_model_len or enable CPU offloading",
                                priority="High",
                                expected_improvement="Better GPU utilization"
                            ))
                else:
                    print("⚠️  No GPU detected - CPU inference will be significantly slower")
                    self.recommendations.append(OptimizationRecommendation(
                        category="Hardware",
                        issue="No GPU detected",
                        recommendation="Install CUDA-compatible GPU for production",
                        priority="Critical",
                        expected_improvement="10-100x performance improvement"
                    ))
            except Exception as e:
                print(f"⚠️  Could not analyze GPU: {e}")
        else:
            print("⚠️  GPUtil not available - install with: pip install gputil")
        
        # CPU Analysis
        cpu_count = psutil.cpu_count()
        cpu_usage = psutil.cpu_percent(interval=1)
        
        print(f"🖥️  CPU: {cpu_count} cores (Usage: {cpu_usage:.1f}%)")
        
        if cpu_count < 8:
            self.recommendations.append(OptimizationRecommendation(
                category="CPU",
                issue="Limited CPU cores",
                recommendation="Upgrade to 8+ cores for better preprocessing",
                priority="Low",
                expected_improvement="Reduced request queuing"
            ))
    
    def generate_optimized_config(self, target_users=10):
        """Generate optimized vLLM configuration based on system resources"""
        
        # Detect system capabilities
        memory_gb = psutil.virtual_memory().total / (1024**3)
        
        gpu_memory_gb = 0
        if GPUTIL_AVAILABLE:
            try:
                gpus = GPUtil.getGPUs()
                gpu_memory_gb = gpus[0].memoryTotal / 1024 if gpus else 0
            except:
                gpu_memory_gb = 0
        
        # Base configuration
        config = {
            "model": "ibm-granite/granite-3.3-8b-instruct",
            "served_model_name": "granite-3.3-8b",
            "host": "0.0.0.0",
            "port": 8000,
            "uvicorn_log_level": "info"
        }
        
        # Memory optimization
        if gpu_memory_gb >= 16:
            config.update({
                "gpu_memory_utilization": 0.9,
                "max_model_len": 8192,
                "max_num_seqs": min(256, target_users * 4)
            })
        elif gpu_memory_gb >= 8:
            config.update({
                "gpu_memory_utilization": 0.85,
                "max_model_len": 4096,
                "max_num_seqs": min(128, target_users * 2)
            })
        else:
            config.update({
                "gpu_memory_utilization": 0.8,
                "max_model_len": 2048,
                "max_num_seqs": min(64, target_users)
            })
        
        # Performance optimizations
        config.update({
            "enable_prefix_caching": True,
            "disable_log_stats": False,
            "tensor_parallel_size": 1,  # Single GPU
            "pipeline_parallel_size": 1,
            "worker_use_ray": False
        })
        
        # Add swap space if system has sufficient RAM
        if memory_gb >= 32:
            config["swap_space"] = min(8, int(memory_gb * 0.2))  # 20% of RAM or 8GB max
        
        return config
    
    def print_recommendations(self):
        """Print optimization recommendations"""
        if not self.recommendations:
            print("✅ No critical optimization issues detected!")
            return
        
        print("\n🎯 Optimization Recommendations")
        print("=" * 60)
        
        # Sort by priority
        priority_order = {"Critical": 0, "High": 1, "Medium": 2, "Low": 3}
        sorted_recs = sorted(self.recommendations, 
                           key=lambda x: priority_order.get(x.priority, 4))
        
        for rec in sorted_recs:
            priority_emoji = {
                "Critical": "🚨",
                "High": "🔴", 
                "Medium": "🟡",
                "Low": "🟢"
            }.get(rec.priority, "⚪")
            
            print(f"\n{priority_emoji} {rec.priority} Priority - {rec.category}")
            print(f"   Issue: {rec.issue}")
            print(f"   Recommendation: {rec.recommendation}")
            print(f"   Expected Improvement: {rec.expected_improvement}")
    
    def generate_startup_script(self, config):
        """Generate production-ready startup script"""
        script = f"""#!/bin/bash
# vLLM Production Startup Script for Granite 3.3 8B
# Generated automatically based on system analysis

# Set environment variables
export CUDA_VISIBLE_DEVICES=0
export TOKENIZERS_PARALLELISM=false

# Start vLLM server with optimized configuration
python -m vllm.entrypoints.openai.api_server \\
    --model {config['model']} \\
    --served-model-name {config['served_model_name']} \\
    --host {config['host']} \\
    --port {config['port']} \\
    --gpu-memory-utilization {config['gpu_memory_utilization']} \\
    --max-model-len {config['max_model_len']} \\
    --max-num-seqs {config['max_num_seqs']} \\
    --enable-prefix-caching \\
    --tensor-parallel-size {config['tensor_parallel_size']} \\
    --pipeline-parallel-size {config['pipeline_parallel_size']}"""
        
        if 'swap_space' in config:
            script += f" \\\n    --swap-space {config['swap_space']}"
        
        script += f" \\\n    --uvicorn-log-level {config['uvicorn_log_level']}"
        
        return script

# Create optimizer instance and run analysis
optimizer = vLLMOptimizer()
optimizer.analyze_system_resources()
optimizer.print_recommendations()

# Generate optimized configuration
print("\n⚙️  Generating Optimized Configuration")
print("=" * 60)
config = optimizer.generate_optimized_config(target_users=10)

print("📋 Recommended vLLM Configuration:")
print(json.dumps(config, indent=2))

print("\n🚀 Production Startup Script:")
startup_script = optimizer.generate_startup_script(config)
print(startup_script)

print("\n💡 Save the startup script as 'start_vllm.sh' and make it executable:")
print("   chmod +x start_vllm.sh")
print("   ./start_vllm.sh")

In [ ]:
# Final Testing Summary & Validation
print("🧪 Notebook Testing Summary")
print("=" * 60)

# Check if all major components are working
components_status = {}

# 1. System Prerequisites
try:
    check_prerequisites()
    components_status['Prerequisites Check'] = "✅ Working"
except Exception as e:
    components_status['Prerequisites Check'] = f"❌ Error: {e}"

# 2. Ollama Integration
try:
    if ollama.is_running():
        components_status['Ollama Integration'] = "✅ Server Running"
    else:
        components_status['Ollama Integration'] = "⚠️  Server Not Running"
except Exception as e:
    components_status['Ollama Integration'] = f"❌ Error: {e}"

# 3. vLLM Components
try:
    from vllm import LLM, SamplingParams
    components_status['vLLM Import'] = "✅ Available"
except Exception as e:
    components_status['vLLM Import'] = f"❌ Error: {e}"

# 4. Benchmarking Tools
try:
    benchmark_ready = hasattr(benchmark, 'sequential_benchmark')
    components_status['Benchmarking Tools'] = "✅ Ready" if benchmark_ready else "❌ Not Ready"
except Exception as e:
    components_status['Benchmarking Tools'] = f"❌ Error: {e}"

# 5. Optimization Analyzer
try:
    optimizer_ready = hasattr(optimizer, 'analyze_system_resources')
    components_status['Optimization Analyzer'] = "✅ Ready" if optimizer_ready else "❌ Not Ready"
except Exception as e:
    components_status['Optimization Analyzer'] = f"❌ Error: {e}"

print("\n📋 Component Status:")
for component, status in components_status.items():
    print(f"   {component}: {status}")

# Summary and recommendations
working_components = sum(1 for status in components_status.values() if status.startswith("✅"))
total_components = len(components_status)

print(f"\n📊 Overall Status: {working_components}/{total_components} components working")

if working_components == total_components:
    print("🎉 All components are working perfectly!")
    print("✅ Ready for production scaling comparison")
elif working_components >= 3:
    print("✅ Core functionality working - ready for basic testing")
    print("💡 Some advanced features may require GPU or additional setup")
else:
    print("⚠️  Several components need attention")
    print("🔧 Check prerequisites and installation")

print("\n🚀 Next Steps:")
if ollama.is_running():
    print("   • Test Ollama performance with your specific workloads")
    print("   • Use benchmarking tools to establish baselines")

if 'vLLM Import' in components_status and components_status['vLLM Import'].startswith("✅"):
    print("   • Set up vLLM server for production testing")
    print("   • Compare performance between Ollama and vLLM")

print("   • Review optimization recommendations")
print("   • Plan your migration strategy based on requirements")

print("\n💫 Happy Scaling!")

In [ ]:
# Quick Ollama Performance Test
if ollama.is_running():
    print("🦙 Quick Ollama Performance Test")
    print("=" * 40)
    
    test_prompt = "Explain the benefits of using Ollama for local AI development in 2-3 sentences."
    result = ollama.test_inference(test_prompt, max_tokens=150)
    
    if result and result['success']:
        print(f"✅ Response time: {result['total_time']:.2f} seconds")
        print(f"🚀 Generation speed: {result['tokens_per_second']:.1f} tokens/sec")
        print(f"📝 Response: {result['response']}")
        
        # Performance analysis
        if result['total_time'] < 5:
            print("🟢 Excellent response time for local inference")
        elif result['total_time'] < 10:
            print("🟡 Good response time for development use")
        else:
            print("🟠 Consider optimizing for production workloads")
            
        if result['tokens_per_second'] > 20:
            print("🟢 Good token generation speed")
        else:
            print("🟡 Token generation could be faster with GPU")
    else:
        print(f"❌ Test failed: {result.get('error', 'Unknown error') if result else 'No response'}")
else:
    print("⚠️  Ollama server not running - start with: ollama serve")

print("\n💡 This demonstrates Ollama's real-world performance on your system!")

## 7. Migration Checklist and Next Steps

### ✅ Pre-Migration Checklist

Before migrating from Ollama to vLLM, ensure you have:

#### **Technical Readiness**
- [ ] **GPU Requirements**: NVIDIA GPU with 8GB+ VRAM
- [ ] **System Resources**: 16GB+ RAM, 8+ CPU cores
- [ ] **CUDA Installation**: Proper CUDA drivers and toolkit
- [ ] **Python Environment**: Python 3.8+ with pip
- [ ] **Network Configuration**: Proper firewall rules for API access

#### **Application Readiness**
- [ ] **API Compatibility**: Code uses OpenAI-compatible endpoints
- [ ] **Error Handling**: Robust error handling for API failures
- [ ] **Load Testing**: Existing load testing infrastructure
- [ ] **Monitoring**: System and application monitoring setup
- [ ] **Backup Plan**: Rollback strategy to Ollama if needed

#### **Operational Readiness**
- [ ] **Documentation**: Updated deployment documentation
- [ ] **Team Training**: Team familiar with vLLM configuration
- [ ] **Monitoring Tools**: GPU monitoring and alerting
- [ ] **Capacity Planning**: Resource requirements calculated
- [ ] **SLA Definition**: Performance SLAs established

### 🗺️ Migration Strategy

| **Phase** | **Duration** | **Activities** | **Success Criteria** |
|-----------|--------------|----------------|---------------------|
| **Phase 1: Setup** | 1-2 days | Install vLLM, basic configuration | Model loads and responds |
| **Phase 2: Testing** | 3-5 days | Performance testing, optimization | Meets performance targets |
| **Phase 3: Integration** | 1-2 weeks | API integration, monitoring | Full application compatibility |
| **Phase 4: Production** | 1 week | Gradual rollout, monitoring | Stable production operation |

### 🎯 Success Metrics

Track these metrics to validate your migration:

#### **Performance Improvements**
- **Throughput**: >3x improvement in requests/second
- **Latency**: Consistent response times under load  
- **Concurrency**: Support for 10+ simultaneous users
- **Resource Efficiency**: 80%+ GPU utilization

#### **Operational Metrics**
- **Uptime**: 99.9%+ availability
- **Error Rate**: <1% API error rate
- **Response Time**: P95 latency <2 seconds
- **Memory Usage**: Stable memory consumption

### 🚀 Next Steps and Advanced Topics

#### **Immediate Next Steps**
1. **Run the benchmarks** in this notebook to establish baselines
2. **Test with your specific workload** and prompts
3. **Implement monitoring** for production deployment
4. **Create deployment automation** with Docker/Kubernetes

#### **Advanced Optimization Topics**
- **Multi-GPU Scaling**: Tensor parallelism for larger deployments
- **Quantization**: Using 4-bit/8-bit quantized models
- **Custom Models**: Fine-tuned Granite models with vLLM
- **Auto-scaling**: Dynamic scaling based on demand
- **Edge Deployment**: Optimizing for edge computing scenarios

#### **Community and Support**
- **vLLM Documentation**: [docs.vllm.ai](https://docs.vllm.ai)
- **GitHub Issues**: [github.com/vllm-project/vllm](https://github.com/vllm-project/vllm)
- **Community Discord**: Join vLLM community discussions
- **IBM Granite Resources**: [ibm.com/granite](https://ibm.com/granite)

### 📚 Additional Resources

- **Production Deployment Guide**: [Advanced vLLM Production Setup](https://docs.vllm.ai/en/latest/serving/production.html)
- **Performance Tuning**: [vLLM Performance Best Practices](https://docs.vllm.ai/en/latest/performance/performance_tips.html)
- **Monitoring Solutions**: Prometheus + Grafana for vLLM metrics
- **Container Deployment**: Docker and Kubernetes configurations

---

## 🎉 Congratulations!

You've successfully learned how to scale from Ollama to vLLM with Granite 3.3 8B! This migration will enable your application to:

- **Handle significantly more concurrent users**
- **Achieve better resource utilization**
- **Maintain consistent performance under load**
- **Scale efficiently as your demands grow**

Remember: The key to successful scaling is **gradual migration** with **thorough testing** at each step. Start with non-critical workloads and gradually increase the scope as you gain confidence with vLLM.

**Happy scaling! 🚀**